In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, accuracy_score
import tensorflow.keras.backend as K

In [ ]:
data_dir = "/kaggle/input/datasets/afridirahman/brain-stroke-ct-image-dataset/Brain_Data_Organised"

In [ ]:
def preprocess_image(img):
    img = img / 255.0
    img = tf.image.adjust_contrast(img, 2)
    img = tf.image.adjust_brightness(img, 0.1)
    return img

In [ ]:
img_size = (224, 224)
batch_size = 32

datagen = ImageDataGenerator(
    preprocessing_function=preprocess_image,
    validation_split=0.2,
    rotation_range=30,
    zoom_range=0.3,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

train_data = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='training'
)

val_data = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

In [ ]:
def focal_loss(gamma=2., alpha=0.75):
    def loss(y_true, y_pred):
        y_pred = K.clip(y_pred, 1e-7, 1 - 1e-7)
        pt = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        return -K.mean(alpha * K.pow(1 - pt, gamma) * K.log(pt))
    return loss

In [ ]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))

# Train all layers (important)
for layer in base_model.layers:
    layer.trainable = True

x = base_model.output
x = GlobalAveragePooling2D()(x)

# Deep Dense
x = Dense(512, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)

x = Dense(128, activation='relu')(x)
x = BatchNormalization()(x)

output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=0.00005),
    loss=focal_loss(),   # 🔥 key improvement
    metrics=['accuracy']
)

model.summary()

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)

lr_reduce = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=3,
    min_lr=1e-6
)

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=50,
    callbacks=[early_stop, lr_reduce]
)

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Reshape, LSTM, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

def build_lstm_model(lstm_units=64, stacked=False):

    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))

    # Freeze layers
    for layer in base_model.layers[:-15]:
        layer.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)

    # Convert to sequence
    x = Reshape((32, -1))(x)

    # LSTM
    if stacked:
        x = LSTM(lstm_units, return_sequences=True)(x)
        x = LSTM(lstm_units)(x)
    else:
        x = LSTM(lstm_units)(x)

    # Dense layers
    x = Dense(256, activation='relu')(x)
    
    x = Dropout(0.5)(x)

    x = Dense(128, activation='relu')(x)

    output = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=base_model.input, outputs=output)

    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
class_weights = {0: 1.0, 1: 2.0}

In [ ]:
model_lstm128 = build_lstm_model(128, stacked=False)

history_lstm128 = model_lstm128.fit(
    train_data,
    validation_data=val_data,
    epochs=50,
    class_weight=class_weights
)

In [ ]:
y_pred_prob = model.predict(val_data)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = val_data.classes

In [ ]:
print(classification_report(y_true, y_pred))

In [ ]:
print("Accuracy:", accuracy_score(y_true, y_pred))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_pred_prob)
roc_auc = auc(fpr, tpr)

print("AUC:", roc_auc)

plt.plot(fpr, tpr, label='AUC = %.3f' % roc_auc)
plt.plot([0,1], [0,1], 'r--')
plt.legend()
plt.title("ROC Curve")
plt.show()

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Accuracy')
plt.legend(['Train', 'Validation'])
plt.show()

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Loss')
plt.legend(['Train', 'Validation'])
plt.show()